In [0]:
dbutils.widgets.text("env", "dev", "env")
dbutils.widgets.text("param_name", "ingestion", "param name")
dbutils.widgets.text("process_name", "all_pending_process", "process name")
dbutils.widgets.text("process_date", "20260716", "process date")

In [0]:
global env, param_name, process_name, process_date,workspace, metadata_schema,execution_type
env = dbutils.widgets.get("env")
param_name = dbutils.widgets.get("param_name")
process_name = dbutils.widgets.get("process_name")
process_date = dbutils.widgets.get("process_date")
process_date = int(process_date)
workspace = 'workspace'
metadata_schema = f"metadata_{env}"
print(env)
print(param_name)
print(process_name)
print(process_date)
print(workspace)
print(metadata_schema)


In [0]:
def get_all_tables(param_name):
    all_process_list = spark.sql(f"""select distinct target_table from {metadata_schema}.dp_metadata_{env} where param_name = '{param_name}'""").toPandas()['target_table'].tolist()
    # print(all_process_list)
    return all_process_list
all_process_list = get_all_tables(param_name)
print(all_process_list)

In [0]:
def get_not_scheduled_tables(param_name,process_date):
    not_scheduled_process_list = spark.sql(f"""SELECT distinct target_table FROM {metadata_schema}.dp_metadata_{env} a inner join {metadata_schema}.dp_calender_{env} b on b.param_name = a.param_name and  b.process_name = a.process_name  where a.param_name = '{param_name}' and b.off_date== {process_date} and b.is_active = 'Y' """).toPandas()['target_table'].tolist()
    return not_scheduled_process_list

not_scheduled_process_list = get_not_scheduled_tables(param_name,process_date)
print(not_scheduled_process_list)

In [0]:
def get_completed_tables(param_name, process_date):
    completed_process_list = spark.sql(f"""select distinct target_table from {metadata_schema}.dp_batch_{env} where param_name = '{param_name}' and process_date = {process_date} and process_status = 'success'""").toPandas()['target_table'].tolist()
    # print(completed_process_list)
    return completed_process_list

completed_process_list= get_completed_tables(param_name, process_date)
print(completed_process_list)

In [0]:
def def_todays_process(all_process_list,not_scheduled_process_list):
    today_process_list = [x for x in all_process_list if x not in not_scheduled_process_list] if all_process_list else []
    return today_process_list
today_process_list = def_todays_process(all_process_list,not_scheduled_process_list)
print(today_process_list)

In [0]:
def check_for_update(today_process_list,completed_process_list):
    if today_process_list==completed_process_list:
        return True
    else:
        return False
update_ind = check_for_update(today_process_list,completed_process_list)
print(update_ind)

In [0]:
def update_process(param_name,param_date):
    try:
        param_date = int(param_date) + 1
        updated_param_date = spark.sql(f"""update {metadata_schema}.param_table_{env} set param_date = {param_date} where param_name = '{param_name}'""")
        return f"param_date updated successfully for {param_name} to {param_date}"
    except Exception as e:
        return e
        pass

In [0]:
if not update_ind:
    result = "No update required"
else:
    result = update_process(param_name, process_date)
dbutils.notebook.exit(result)